# AI Kids Story Studio — One-Click Episode 1 Animation Batch (8 Action Shots, Wan2.2-TI2V-5B)

**How to run (One Click):**
1. In Google Colab, select **Runtime > Change runtime type > A100 GPU** (with **High-RAM** enabled).
2. Click **Runtime > Run all**.
3. Wait for automatic hardware check, dependency setup, generation of **8 I2V MP4 clip(s)** (`hook_blur`, `crossroads_greeting`, `go_signal`, `stone_slip`, `hare_sprints`, `hare_helps`, `crossing_together`, `crossing_finish` at `832x480`, `81` frames @ `16` fps), and automatic result ZIP download.

---
### Budget & License Guardrails
- **Strict 8-clip budget:** Generates **at most 8 clip(s)** (`hook_blur`, `crossroads_greeting`, `go_signal`, `stone_slip`, `hare_sprints`, `hare_helps`, `crossing_together`, `crossing_finish`), loading the model once, with zero automatic retries, resumable clip outputs if persistent storage is enabled, and automatic partial ZIP export if a later clip fails.
- **Model & License:** [`Wan-AI/Wan2.2-TI2V-5B-Diffusers`](https://huggingface.co/Wan-AI/Wan2.2-TI2V-5B-Diffusers) (pinned commit `b8fff7315c768468a5333511427288870b2e9635`), **apache-2.0** ([View LICENSE.txt](https://huggingface.co/Wan-AI/Wan2.2-TI2V-5B-Diffusers/blob/b8fff7315c768468a5333511427288870b2e9635/LICENSE.txt)). Ungated public weights.
- **Candidate status:** Generated MP4s are **unapproved candidates** until inspected and explicitly approved locally via `AssetStore`.


In [ ]:
# 1. Optional persistent storage settings (Defaults require ZERO edits: temporary Colab storage, full auto-run).
USE_GOOGLE_DRIVE = False
PERSISTENT_DIR = ""

EXPECTED_PACKAGE_SHA256 = "fe1162b4d65e58e67814f80c8c548a9a3400bf3acf54ada4ef06acefeb986f81"
EXPECTED_REQUEST_SHA256 = "4b0ac5054e7a5ab3d9ebc1957508a703b35b6b6136d9c4623cade5500cb02075"
EXPECTED_PROFILE_SHA256 = "d58fe593b5a38087cc4e2f94cf9050fecc92a4fee923483f0c930cf15850305c"
REMOTE_PACKAGE_URLS = (
    "https://raw.githubusercontent.com/SandeshAwate6055/quiet-meteor-73/b5e21e6550c00f7bc3bdc3e8b738a0e39f69427c/examples/episode_01_i2v_v1_one_click.zip",
    "https://raw.githubusercontent.com/SandeshAwate6055/quiet-meteor-73/master/examples/episode_01_i2v_v1_one_click.zip",
)
MIN_VRAM_GIB = 22
MIN_RAM_GIB = 24
MIN_DISK_GIB = 32


In [ ]:
# 2. Immediate GPU, VRAM, host RAM and disk preflight BEFORE any dependency install or model download.
from pathlib import Path
import shutil
import torch

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_ROOT = Path(PERSISTENT_DIR or "/content/drive/MyDrive/AI_Kids_Story_Studio/one_click_i2v")
else:
    WORK_ROOT = Path(PERSISTENT_DIR or "/content/story_one_click_i2v")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

cuda_ok = torch.cuda.is_available()
bf16_ok = bool(cuda_ok and torch.cuda.get_device_capability(0)[0] >= 8)
vram_gib = (torch.cuda.get_device_properties(0).total_memory / 2**30) if cuda_ok else 0.0
gpu_name = torch.cuda.get_device_name(0) if cuda_ok else "NONE"
meminfo = Path("/proc/meminfo")
ram_gib = (
    int(next(line.split()[1] for line in meminfo.read_text().splitlines() if line.startswith("MemTotal:"))) / 1024**2
    if meminfo.exists() else 0.0
)
disk_gib = shutil.disk_usage(WORK_ROOT).free / 2**30

print(f"Detected GPU: {gpu_name} | CUDA: {cuda_ok} | BF16: {bf16_ok} | VRAM: {vram_gib:.2f} GiB | Host RAM: {ram_gib:.2f} GiB | Free Disk: {disk_gib:.2f} GiB")
instruction = "Select Runtime > Change runtime type > L4 or A100 GPU with High-RAM, then click Runtime > Run all."
if not cuda_ok or not bf16_ok or vram_gib < MIN_VRAM_GIB:
    raise RuntimeError(f"No compatible CUDA/BF16 GPU (need >= {MIN_VRAM_GIB} GiB VRAM, got {vram_gib:.2f} GiB on {gpu_name}). " + instruction)
if ram_gib < MIN_RAM_GIB:
    raise RuntimeError(f"Insufficient host RAM (need >= {MIN_RAM_GIB} GiB, got {ram_gib:.2f} GiB). " + instruction)
if disk_gib < MIN_DISK_GIB:
    raise RuntimeError(f"Insufficient free disk space (need >= {MIN_DISK_GIB} GiB, got {disk_gib:.2f} GiB).")


In [ ]:
# 3. Download and SHA256-verify the I2V handoff package, extract worker modules, and check hardware profile.
import hashlib, json, sys, urllib.request, zipfile
from pathlib import Path

package_path = WORK_ROOT / "episode_01_i2v_v1_one_click_fe1162b4d65e.zip"
if not package_path.exists() or hashlib.sha256(package_path.read_bytes()).hexdigest() != EXPECTED_PACKAGE_SHA256:
    payload = None
    fetch_errors = []
    for url in REMOTE_PACKAGE_URLS:
        try:
            with urllib.request.urlopen(url, timeout=30) as resp:
                candidate = resp.read()
            actual_sha = hashlib.sha256(candidate).hexdigest()
            if actual_sha != EXPECTED_PACKAGE_SHA256:
                fetch_errors.append(f"{url}: SHA256 mismatch (expected {EXPECTED_PACKAGE_SHA256}, got {actual_sha})")
                continue
            payload = candidate
            print("Retrieved and SHA256-verified I2V handoff package from:", url)
            break
        except Exception as exc:
            fetch_errors.append(f"{url}: {exc}")
    if payload is None:
        raise RuntimeError("Could not download SHA256-verified I2V package: " + "; ".join(fetch_errors))
    package_path.write_bytes(payload)

bootstrap_dir = WORK_ROOT / "bundles" / EXPECTED_PACKAGE_SHA256
with zipfile.ZipFile(package_path) as zf:
    index = json.loads(zf.read("bundle_index.json"))
    for mod in ("__init__.py", "asset_store.py", "characters.py", "schema.py",
                "production_state.py", "modern_video.py", "one_click_video.py"):
        member = f"src/{mod}"
        data = zf.read(member)
        if hashlib.sha256(data).hexdigest() != index["files"][member]:
            raise ValueError(f"Bootstrap module checksum mismatch: {member}")
        target = bootstrap_dir / member
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            target.write_bytes(data)

if str(bootstrap_dir) not in sys.path:
    sys.path.insert(0, str(bootstrap_dir))

from src.one_click_video import extract_and_verify_i2v_bundle
from src.modern_video import check_i2v_hardware

bundle_info = extract_and_verify_i2v_bundle(
    package_path, WORK_ROOT / "bundles", expected_package_sha256=EXPECTED_PACKAGE_SHA256
)
BUNDLE_DIR = bundle_info["bundle_dir"]
PROFILE = bundle_info["profile"]
check_i2v_hardware(PROFILE, cuda=cuda_ok, bf16=bf16_ok, vram_gib=vram_gib, ram_gib=ram_gib, disk_gib=disk_gib)
print("Verified I2V bundle:", BUNDLE_DIR)
print("Model:", PROFILE["model_id"], "| Revision:", PROFILE["model_revision"], "| License:", PROFILE["model_license"])


In [ ]:
# 4. Install missing Colab dependencies once (cached across reruns in the same runtime).
import importlib.metadata, subprocess, sys

req_file = BUNDLE_DIR / "requirements-one-click.txt"
required_pins = {}
for line in req_file.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if line and not line.startswith("#") and "==" in line:
        pkg, ver = line.split("==", 1)
        required_pins[pkg.strip()] = ver.strip()

missing = []
for pkg, ver in required_pins.items():
    try:
        if importlib.metadata.version(pkg) != ver:
            missing.append(f"{pkg}=={ver}")
    except importlib.metadata.PackageNotFoundError:
        missing.append(f"{pkg}=={ver}")
for extra in ("ftfy", "imageio", "imageio-ffmpeg"):
    try:
        importlib.metadata.version(extra)
    except importlib.metadata.PackageNotFoundError:
        missing.append(extra)

if missing:
    print("Installing I2V dependencies:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
    importlib.invalidate_caches()
else:
    print("All I2V dependencies already installed.")


In [ ]:
# 5. Load Wan2.2-TI2V-5B once, animate ONLY 'hare_sprints' (81 frames @ 16 fps), preview & auto-download ZIP.
import json
from IPython.display import Video, display
from google.colab import files
from src.modern_video import WanI2VBackend
from src.one_click_video import run_one_click_i2v_batch

if "_ONE_CLICK_I2V_BACKEND" not in globals():
    _ONE_CLICK_I2V_BACKEND = None

CACHE_DIR = WORK_ROOT / "model_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def get_backend(worker_settings, previous):
    global _ONE_CLICK_I2V_BACKEND
    if _ONE_CLICK_I2V_BACKEND is not None:
        if _ONE_CLICK_I2V_BACKEND.settings != worker_settings:
            raise RuntimeError("I2V worker settings changed; restart runtime.")
        return _ONE_CLICK_I2V_BACKEND
    _ONE_CLICK_I2V_BACKEND = WanI2VBackend(worker_settings, PROFILE, CACHE_DIR, previous)
    return _ONE_CLICK_I2V_BACKEND

run_result = run_one_click_i2v_batch(BUNDLE_DIR, WORK_ROOT, backend_factory=get_backend, cache_dir=CACHE_DIR, allow_partial=True)
print("I2V batch outcome:", json.dumps(run_result["outcome"], indent=2))
for asset in run_result["outcome"]["assets"]:
    clip_path = run_result["result_dir"] / asset["file"]
    print(f"Generated {asset['kind']}/{asset['target_id']} | sha256={asset['sha256'][:16]}")
    display(Video(str(clip_path), embed=True))

print("Exported result ZIP:", run_result["archive_path"], "| SHA256:", run_result["archive_sha256"])
files.download(str(run_result["archive_path"]))
if run_result["outcome"].get("status") != "complete":
    raise RuntimeError(f"I2V batch did not complete ({run_result['outcome'].get('status')}): {run_result['outcome'].get('errors')}")
